# Vaidya — Phase 2: QLoRA Training (Kaggle T4 ×2)

**Before running:**
1. Settings → Accelerator → **GPU T4 ×2** ✅
2. Settings → Internet → **On** ✅
3. Add-ons → Secrets → add `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`
4. Upload `train.py` and `qlora_config.py` as a Kaggle Dataset, or paste them into cells below

**Expected runtime:** ~5–7 hours on T4 ×2 (1 epoch, 96k sequences, seq_len=640)

> **Note:** T4 is CC 7.5 — no BF16. Training uses FP16 automatically.  
> `device_map="auto"` spreads the 4-bit model across both T4s (~7 GB each).

## Step 1 — Install packages

In [ ]:
# Pinned versions: stable, compatible, support Python 3.13 + CUDA 12.8
# DO NOT use --upgrade — it pulls TRL 1.x / transformers 5.x which have breaking changes
!pip install -q \
    transformers==4.47.0 \
    peft==0.14.0 \
    trl==0.15.2 \
    bitsandbytes==0.45.3 \
    accelerate==0.35.0 \
    datasets==3.2.0 \
    mlflow==2.19.0 \
    boto3 \
    liger-kernel
print('✅ Install done — RESTART THE KERNEL (Run → Restart Session), then re-run all cells.')

## Step 2 — Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Enable GPU in Notebook Settings.'
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
cc   = torch.cuda.get_device_capability()
print(f'GPU  : {gpu}')
print(f'VRAM : {vram:.1f} GB')
print(f'CC   : {cc[0]}.{cc[1]}')
print(f'BF16 supported: {torch.cuda.is_bf16_supported()}')
# P100 → CC 6.0, ~16 GB, bf16=False → training will use fp16

## Step 3 — Pull training scripts from GitHub

In [ ]:
import os

os.chdir('/kaggle/working')

# Always pull latest train.py + qlora_config.py from GitHub
!git clone --depth 1 https://github.com/Anand09-in/vaidya.git _repo 2>&1 | tail -2
import shutil
for f in ['train.py', 'qlora_config.py']:
    shutil.copy(f'_repo/layer2_finetune/{f}', f'./{f}')
    print(f'✅  {f}')

!rm -rf _repo

In [ ]:
# Verify scripts are present
for f in ['train.py', 'qlora_config.py']:
    status = '✅' if os.path.exists(f) else '❌ MISSING'
    print(f'{status}  {f}')

In [ ]:
# Verify which version of the scripts is loaded
import importlib.util, sys

for fname, modname in [('train.py', 'train_mod'), ('qlora_config.py', 'qlora_config')]:
    spec = importlib.util.spec_from_file_location(modname, f'/kaggle/working/{fname}')
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    print(f"{fname}  →  version {getattr(mod, '__version__', 'MISSING')}")

## Step 4 — Pull data from S3

In [ ]:
from kaggle_secrets import UserSecretsClient
import os, boto3

s = UserSecretsClient()
os.environ['AWS_ACCESS_KEY_ID']     = s.get_secret('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = s.get_secret('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION']    = s.get_secret('AWS_DEFAULT_REGION')

# Use boto3 directly — avoids aws CLI PATH issues on Kaggle
os.makedirs('./data', exist_ok=True)
s3 = boto3.client('s3')

paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket='vaidya-artifacts', Prefix='data/'):
    for obj in page.get('Contents', []):
        key = obj['Key']
        if key.endswith('.parquet'):
            fname = key.split('/')[-1]
            local = f'./data/{fname}'
            print(f'Downloading {key} → {local}')
            s3.download_file('vaidya-artifacts', key, local)

import pandas as pd
for split in ['train', 'val', 'test']:
    df = pd.read_parquet(f'./data/{split}.parquet')
    print(f'{split}: {len(df):,} rows')

## Step 5 — Run training

Uses `python train.py` directly (no Accelerate/DeepSpeed on Kaggle — single process is simplest).  
Training logs stream to the cell output. Expected: ~6–8 hours.

In [ ]:
# Dry run — check the script loads and prints config without training
!python train.py --help

In [ ]:
# Pre-flight: verify all packages are importable before starting the 6-hour run
import importlib
required = ['transformers', 'peft', 'trl', 'bitsandbytes', 'accelerate', 'datasets', 'mlflow', 'boto3']
all_ok = True
for pkg in required:
    try:
        m = importlib.import_module(pkg)
        print(f'✅  {pkg} {getattr(m, "__version__", "?")}')
    except ImportError as e:
        print(f'❌  {pkg}  — {e}')
        all_ok = False

if not all_ok:
    print('\n⚠️  Re-run Step 1 (install cell), then restart the kernel and run all cells again.')
else:
    print('\n✅ Environment ready — proceed to training.')

In [ ]:
# ⚡ FULL TRAINING RUN — takes 6–8 hours
# Change --run-name to run2, run3 etc. for subsequent hyperparameter runs
# If OOM: add --batch-size 1 --grad-accum 32

!python train.py --run-name run1

## Step 6 — Verify checkpoint on S3

In [ ]:
import boto3

s3 = boto3.client('s3')

print('=== Checkpoint files ===')
resp = s3.list_objects_v2(Bucket='vaidya-artifacts', Prefix='checkpoints/run1/')
for obj in resp.get('Contents', []):
    size_mb = obj['Size'] / 1e6
    print(f"  {obj['Key']}  ({size_mb:.1f} MB)")

print()
print('=== MLflow ===')
resp2 = s3.list_objects_v2(Bucket='vaidya-artifacts', Prefix='mlflow/')
for obj in resp2.get('Contents', []):
    print(f"  {obj['Key']}")

## Step 7 — Quick eval (loss sanity check)

Full accuracy eval is Phase 3. Here we just confirm loss decreased.

In [ ]:
import json, glob

# Read trainer_state.json from best checkpoint
states = glob.glob('./checkpoints/run1/checkpoint-*/trainer_state.json')
if not states:
    states = glob.glob('./checkpoints/run1/trainer_state.json')

if states:
    with open(sorted(states)[-1]) as f:
        state = json.load(f)
    log_history = state.get('log_history', [])
    train_losses = [(e['step'], e['loss']) for e in log_history if 'loss' in e]
    eval_losses  = [(e['step'], e['eval_loss']) for e in log_history if 'eval_loss' in e]
    if train_losses:
        print(f'Train loss — first: {train_losses[0][1]:.4f}  last: {train_losses[-1][1]:.4f}')
    if eval_losses:
        print(f'Eval  loss — first: {eval_losses[0][1]:.4f}  last: {eval_losses[-1][1]:.4f}')
else:
    print('No trainer_state.json found — check checkpoints/ directory.')

## Gate check

Before moving to Phase 3, confirm:
- ✅ Training loss decreased (not diverged)
- ✅ Eval loss decreased and didn't spike at the end
- ✅ Checkpoint at `s3://vaidya-artifacts/checkpoints/run1/`
- ✅ MLflow db at `s3://vaidya-artifacts/mlflow/mlflow.db`